# Task C: From Hidden Count Feature to Direct Count Output

这个 notebook 对应核心实验 C：模型是否先形成 internal counting feature，随后再把这个 feature 转换成最终输出的 `NUM_k` answer token。

当前版本是一个最小可运行 smoke test：用 `NeedleCount-synthetic` 做 single-token direct count。每条序列里随机放入 `k` 个 NEEDLE token，`k` 在 `0..max_count` 上均匀采样；末尾追加 query token，模型只在 query 位置预测一个答案 token `NUM_k`。

## Read This First

- 这次只跑了一个 smoke setting：`seq_len=128`, `max_count=10`, `noise_vocab=32`, `steps=1000`, `seed=0`。
- 末轮 answer accuracy 是 `1.000`，best-count-layer R2 是 `0.998`，best logit-lens accuracy 是 `1.000`。
- 因为这是单 setting smoke test，结论只用于确认实验 C pipeline 能跑通；正式结论还需要扫 `context length / max count / distractor ratio / seed`。

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'results').exists():
    ROOT = ROOT.parent
RESULT_DIR = ROOT / 'results' / 'needle_count_task_c_direct_count'
FIGURE_DIR = ROOT / 'figures' / 'needle_count_task_c_direct_count'
summary = pd.read_csv(RESULT_DIR / 'summary.csv')
checkpoint = pd.read_csv(RESULT_DIR / 'checkpoint_metrics.csv')
probe = pd.read_csv(RESULT_DIR / 'probe_readout.csv')
steering = pd.read_csv(RESULT_DIR / 'steering.csv')
summary

## 1. 实验问题

Task A/B 已经说明 hidden state 里可以出现可线性读出的 counting feature。实验 C 问的是下一步：这个 feature 是否真的接到输出头，能否直接变成答案 token。

因此这里不只看 `hidden -> count` probe，而是同时看四件事：

1. 模型最终能否输出正确的 `NUM_k`。
2. hidden count probe 是否已经可读。
3. 同一个 hidden 直接过 final output head 时，logit lens 是否已经能读出 `NUM_k`。
4. ridge count direction 是否和 `NUM_k -> NUM_{k+1}` 的 unembedding 方向对齐，并且能否被 steering 系统性推动。

## 2. Task 与训练设置

| item | value |
| --- | --- |
| source task | NeedleCount-synthetic direct final count |
| sequence format | <seq> <QUERY_COUNT> <NUM_k> |
| seq_len | 128 |
| max_count | 10 |
| answer tokens | NUM_0..NUM_10 |
| noise/distractor vocab | 32 |
| model | 3 layers, d_model=128, heads=4 |
| training | 1000 steps, batch=128, lr=0.0003 |
| checkpoints | [0, 50, 100, 200, 500, 1000] |

## 3. 指标定义

| metric | definition | interpretation |
| --- | --- | --- |
| answer_acc | full-vocab top-1 是否等于真实答案 token `NUM_k`。 | 真正的 direct-count 行为准确率。 |
| num_restricted_acc | 只在 `NUM_0..NUM_K` answer token 之间取 argmax 后是否等于真实 k。 | 排除输出非答案 token 后，答案排序本身是否正确。 |
| count_r2 | 在某层 query-position hidden 上训练 ridge regression 预测真实 count 的 test R2。 | hidden 中 count 是否线性可读。 |
| ridge_round_acc | ridge 预测 count 后四舍五入并裁剪到合法范围的准确率。 | 线性 scalar readout 能否直接当作 count 使用。 |
| linear_answer_acc | 同一 hidden 上训练 one-vs-all linear ridge classifier 预测 `NUM_k`。 | 如果 learned readout 高但 model answer 低，瓶颈在模型自己的输出头/readout。 |
| logit_lens_acc | 把中间层 query hidden 直接过 final LN 和 output head，限制在 `NUM_k` 上取 argmax。 | 当前层 hidden 是否已经处在输出头能读的坐标系里。 |
| unembedding_adjacent_cosine | ridge count direction 与平均 `W[NUM_{k+1}] - W[NUM_k]` 的 cosine。 | count feature 是否沿着答案 token 序列的 unembedding 方向排列。 |
| mean_answer_margin | `logit(NUM_true) - max_wrong_NUM_logit`。 | 正确答案相对其他数字 token 的安全边际。 |

## 4. Main Result

最终 checkpoint 的 full-vocab answer accuracy 为 `1.000`，NUM-restricted accuracy 为 `1.000`。best-count layer `1` 的 count R2 为 `0.998`；best logit-lens layer `2` 的 logit-lens accuracy 为 `1.000`。final layer 的 count R2 是 `0.997`，logit-lens accuracy 是 `1.000`。

这张表把行为和 best-layer hidden/readout 指标放在同一时间轴上：

| step | answer_acc | num_restricted_acc | mae | mean_answer_margin | best_count_layer | count_r2 | best_count_ridge_round_acc | best_logit_lens_layer | logit_lens_acc | best_linear_layer | linear_answer_acc |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 0 | 0.1021 | 3.194 | -0.6627 | 0 | 0.9534 | 0.5649 | 0 | 0.0957 | 1 | 0.2358 |
| 50 | 0.3135 | 0.3135 | 1.49 | -0.2094 | 2 | 0.9594 | 0.5806 | 2 | 0.313 | 2 | 0.4014 |
| 100 | 0.6729 | 0.6729 | 0.3945 | 0.2652 | 2 | 0.9897 | 0.8872 | 2 | 0.6279 | 2 | 0.7646 |
| 200 | 0.9482 | 0.9482 | 0.0518 | 1.336 | 2 | 0.9965 | 0.9888 | 2 | 0.937 | 2 | 0.9878 |
| 500 | 0.9775 | 0.9775 | 0.0225 | 3.326 | 1 | 0.9978 | 1 | 2 | 0.981 | 2 | 1 |
| 1000 | 1 | 1 | 0 | 7.907 | 1 | 0.9975 | 1 | 2 | 1 | 2 | 1 |

![Task C training dynamics](../../figures/needle_count_task_c_direct_count/task_c_training_dynamics.png)

### 4.1 Checkpoint-level behavior

| step | train_loss | loss | answer_acc | num_restricted_acc | off_by_one_rate | mae | bias_pred_minus_true | mean_answer_margin | non_num_top1_rate | non_num_mass |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | NA | 4.082 | 0 | 0.1021 | 0.1846 | 3.194 | -1.04 | -0.6627 | 1 | 0.7929 |
| 50 | 1.982 | 1.985 | 0.3135 | 0.3135 | 0.2925 | 1.49 | 1.171 | -0.2094 | 0 | 0.0705 |
| 100 | 1.077 | 1.05 | 0.6729 | 0.6729 | 0.2598 | 0.3945 | 0.252 | 0.2652 | 0 | 0.0613 |
| 200 | 0.4002 | 0.4109 | 0.9482 | 0.9482 | 0.0518 | 0.0518 | 0.0479 | 1.336 | 0 | 0.0378 |
| 500 | 0.0563 | 0.1233 | 0.9775 | 0.9775 | 0.0225 | 0.0225 | -0.0225 | 3.326 | 0 | 0.0105 |
| 1000 | 0.0021 | 0.0021 | 1 | 1 | 0 | 0 | 0 | 7.907 | 0 | 0.001 |

### 4.2 Final layer-wise readout

这一节看最终 checkpoint 每一层 query-position hidden 的状态。`count_r2 / ridge_round_acc / linear_answer_acc` 是新训练的 probe/readout；`logit_lens_acc` 和 `unembedding_adjacent_cosine` 则直接检查模型自己的 output head 是否能读。

| layer | count_r2 | count_mae | ridge_round_acc | linear_answer_acc | logit_lens_acc | logit_lens_margin | unembedding_adjacent_cosine |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 0.9939 | 0.1963 | 0.959 | 0.4136 | 0.2861 | -0.3683 | 0.03 |
| 1 | 0.9975 | 0.1291 | 1 | 0.7451 | 0.2661 | -0.2944 | 0.235 |
| 2 | 0.9967 | 0.1371 | 0.9844 | 1 | 1 | 7.904 | 0.4167 |

![Task C layer-wise readout](../../figures/needle_count_task_c_direct_count/task_c_layerwise_readout.png)

## 5. Final Query-Position Steering

这里在 final checkpoint 的 best probe layer `2` 上做 query-position steering：`h' = h + beta * std(axis) * normalize(w_count)`，然后直接过 output head 看 `NUM_k` logits。

- baseline beta=0 的 answer accuracy 是 `1.000`，mean answer margin 是 `7.912`。
- 最大正向 beta 的平均预测 count shift 是 `0.000`。
- 最大负向 beta 的平均预测 count shift 是 `0.000`。

注意：这是 final query hidden 的 logit-level steering，不是完整 forward 中替换早层 activation 后继续跑后续层。它能检验 output head 是否沿 count direction 排列，但还不能单独证明早层 count direction 在完整 computation 中因果控制答案。

![Task C final query steering](../../figures/needle_count_task_c_direct_count/task_c_final_query_steering.png)

### 5.1 Steering table

| layer | beta_axis_std | answer_acc | mean_pred_count | mean_pred_count_delta_vs_baseline | frac_pred_plus_one_vs_baseline | frac_pred_minus_one_vs_baseline | mean_answer_margin | non_num_mass |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2 | -5 | 1 | 5.051 | 0 | 0 | 0 | 6.591 | 0.0161 |
| 2 | -3 | 1 | 5.051 | 0 | 0 | 0 | 7.495 | 0.0042 |
| 2 | -1 | 1 | 5.051 | 0 | 0 | 0 | 7.921 | 0.0014 |
| 2 | -0.5 | 1 | 5.051 | 0 | 0 | 0 | 7.923 | 0.0012 |
| 2 | 0 | 1 | 5.051 | 0 | 0 | 0 | 7.912 | 0.001 |
| 2 | 0.5 | 1 | 5.051 | 0 | 0 | 0 | 7.894 | 0.0009 |
| 2 | 1 | 1 | 5.051 | 0 | 0 | 0 | 7.859 | 0.0008 |
| 2 | 3 | 1 | 5.051 | 0 | 0 | 0 | 7.221 | 0.0009 |
| 2 | 5 | 1 | 5.051 | 0 | 0 | 0 | 6.341 | 0.0013 |

## 6. 当前结论

当前 smoke setting 已经同时学会 hidden count 和 direct answer readout。

更具体地说，最终 best-count layer 是 `1`，count R2=`0.998`，ridge round acc=`1.000`；best-output/logit-lens layer 是 `2`，logit-lens acc=`1.000`，linear answer acc=`1.000`。模型真实 answer acc=`1.000`，best-output learned readout 与模型真实输出的 gap 是 `0.000`。

一个关键现象是 step 0 时 best count R2 已经有 `0.953`，但 answer acc 只有 `0.000`。这说明在这个 NeedleCount smoke task 里，随机初始化的 query hidden 已经能线性反映 token occurrence count；训练真正完成的是把这个可读 count 搬到 output head 能稳定读出的 final-layer answer-token 坐标系。

steering 方向检查显示：正向最大平均 count shift 为 `0.000`，负向最大平均 count shift 为 `0.000`。如果这两个数方向稳定且 non-NUM mass 没明显上升，说明 output head 至少在 query hidden 层面使用了 count axis。

## 7. 下一步

1. 扩展正式 grid：`seq_len=[128,512,2000]`、`max_count=[10,30]`、多 seed。
2. 加高 distractor ratio / sparse occurrence setting，测试是否出现类似 Task A 的 hidden-readable 但 output-failed regime。
3. 把 direct count task 接入正式 sampler/config/checkpoint pipeline，和 A/B 的 Transformer baseline 保持完全同构。
4. 做真正 layer-wise activation patch：在早层替换或沿 count direction steering 后继续跑后续层，而不是只做 final query logit steering。
5. 对比 DyckCounter direct count 与 NeedleCount direct count，区分 formal stack counter 和 occurrence retrieval/counting。

In [ ]:
# 可选：重新跑这个 smoke test 并重写 notebook
# !python scripts/task_c_direct_count_readout.py --steps 1000 --checkpoint-steps 0,50,100,200,500,1000
# !python scripts/write_task_c_direct_count_notebook.py